# T10 — Stage III diagnostics module, manual check

Scope (per `tasks.md` T10): a small, reusable, independently tested module
(`iste.diagnostics`) computing `total_particles`, `valid_particles`, `invalid_particles`,
`total_weight`, `effective_sample_size`, and `valid_particle_fraction` from explicit
`weights`/`valid` arrays. This notebook shows tiny hand-computable arrays, the expected
value worked out by hand, and the actual function output, side by side.

This module does **not** run trajectories or recompute transport physics — it is
diagnostics-only, operating on arrays a caller (T09's `iste.ensemble_run`, or later T13)
has already produced.

In [1]:
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import math
import numpy as np

from iste.diagnostics import compute_stage3_diagnostics, compute_stage3_diagnostics_from_log_weights

## 1. All valid, unequal weights

`weights = [1, 3]`, all valid.

By hand:
```
total_particles = 2
valid_particles = 2, invalid_particles = 0
total_weight = 1 + 3 = 4
ESS = (sum w)^2 / sum(w^2) = 4^2 / (1^2 + 3^2) = 16 / 10 = 1.6
valid_particle_fraction = 2/2 = 1.0
```

In [2]:
weights_a = np.array([1.0, 3.0])
valid_a = np.array([True, True])
diag_a = compute_stage3_diagnostics(weights_a, valid_a)

expected_a = dict(total_particles=2, valid_particles=2, invalid_particles=0,
                   total_weight=4.0, effective_sample_size=1.6, valid_particle_fraction=1.0)
print("expected:", expected_a)
print("actual:  ", diag_a)
for field, exp in expected_a.items():
    assert math.isclose(getattr(diag_a, field), exp), field
print("OK -- matches hand calculation")

expected: {'total_particles': 2, 'valid_particles': 2, 'invalid_particles': 0, 'total_weight': 4.0, 'effective_sample_size': 1.6, 'valid_particle_fraction': 1.0}
actual:   StageIIIDiagnostics(total_particles=2, valid_particles=2, invalid_particles=0, total_weight=4.0, effective_sample_size=1.6, valid_particle_fraction=1.0)
OK -- matches hand calculation


## 2. Mixed valid/invalid

`weights = [2, 3, 5]`, `valid = [True, False, True]` — the middle particle (weight 3) is
invalid, so **only 2 and 5 contribute** to `total_weight`/`effective_sample_size`.

By hand:
```
total_particles = 3
valid_particles = 2, invalid_particles = 1
total_weight = 2 + 5 = 7            (the invalid particle's weight 3 is excluded)
ESS = 7^2 / (2^2 + 5^2) = 49 / 29 ≈ 1.6897
valid_particle_fraction = 2/3 ≈ 0.6667
```

In [3]:
weights_b = np.array([2.0, 3.0, 5.0])
valid_b = np.array([True, False, True])
diag_b = compute_stage3_diagnostics(weights_b, valid_b)

expected_b = dict(total_particles=3, valid_particles=2, invalid_particles=1,
                   total_weight=7.0, effective_sample_size=49.0 / 29.0,
                   valid_particle_fraction=2.0 / 3.0)
print("expected:", expected_b)
print("actual:  ", diag_b)
for field, exp in expected_b.items():
    assert math.isclose(getattr(diag_b, field), exp), field
print("OK -- the invalid particle's weight (3) never entered total_weight or ESS")

expected: {'total_particles': 3, 'valid_particles': 2, 'invalid_particles': 1, 'total_weight': 7.0, 'effective_sample_size': 1.6896551724137931, 'valid_particle_fraction': 0.6666666666666666}
actual:   StageIIIDiagnostics(total_particles=3, valid_particles=2, invalid_particles=1, total_weight=7.0, effective_sample_size=1.6896551724137931, valid_particle_fraction=0.6666666666666666)
OK -- the invalid particle's weight (3) never entered total_weight or ESS


## 3. ESS calculation shown explicitly (all valid, four particles)

`weights = [1, 1, 1, 1]` — equal weights, the textbook case where ESS should equal the
number of particles (no concentration).

In [4]:
weights_c = np.array([1.0, 1.0, 1.0, 1.0])
valid_c = np.array([True, True, True, True])

sum_w = np.sum(weights_c[valid_c])
sum_w2 = np.sum(weights_c[valid_c] ** 2)
ess_by_hand = sum_w ** 2 / sum_w2
print(f"sum(w) = {sum_w}, sum(w^2) = {sum_w2}, ESS = (sum w)^2 / sum(w^2) = {ess_by_hand}")

diag_c = compute_stage3_diagnostics(weights_c, valid_c)
print("module output:", diag_c)
assert math.isclose(diag_c.effective_sample_size, ess_by_hand)
assert math.isclose(diag_c.effective_sample_size, 4.0)
print("OK -- equal weights give ESS == N")

sum(w) = 4.0, sum(w^2) = 4.0, ESS = (sum w)^2 / sum(w^2) = 4.0
module output: StageIIIDiagnostics(total_particles=4, valid_particles=4, invalid_particles=0, total_weight=4.0, effective_sample_size=4.0, valid_particle_fraction=1.0)
OK -- equal weights give ESS == N


## 4. Edge cases

Empty input, all-invalid input, zero total weight, and a non-finite weight on an invalid
vs. a valid particle.

In [5]:
# empty input -- valid_particle_fraction is NaN (0/0 is genuinely undefined)
diag_empty = compute_stage3_diagnostics(np.array([]), np.array([], dtype=bool))
print("empty:", diag_empty)
assert diag_empty.total_particles == 0
assert math.isnan(diag_empty.valid_particle_fraction)

# all invalid -- total_weight/ESS are 0.0, valid_particle_fraction is 0.0 (well-defined)
diag_all_invalid = compute_stage3_diagnostics(np.array([1.0, 2.0]), np.array([False, False]))
print("all invalid:", diag_all_invalid)
assert diag_all_invalid.total_weight == 0.0
assert diag_all_invalid.effective_sample_size == 0.0
assert diag_all_invalid.valid_particle_fraction == 0.0

# zero total weight among valid particles -- guarded explicitly, no div-by-zero NaN
diag_zero_w = compute_stage3_diagnostics(np.array([0.0, 0.0]), np.array([True, True]))
print("zero weight:", diag_zero_w)
assert diag_zero_w.effective_sample_size == 0.0

# non-finite weight on an INVALID particle -- ignored, no error
diag_nonfinite_invalid = compute_stage3_diagnostics(np.array([2.0, np.nan]), np.array([True, False]))
print("non-finite on invalid particle (ignored):", diag_nonfinite_invalid)
assert diag_nonfinite_invalid.total_weight == 2.0

# non-finite weight on a VALID particle -- must raise, never silently absorbed
try:
    compute_stage3_diagnostics(np.array([1.0, np.nan]), np.array([True, True]))
    raise AssertionError("expected ValueError")
except ValueError as exc:
    print("non-finite on valid particle correctly raised:", exc)

print("\nOK -- all edge cases behave as documented in iste/diagnostics.py")

empty: StageIIIDiagnostics(total_particles=0, valid_particles=0, invalid_particles=0, total_weight=0.0, effective_sample_size=0.0, valid_particle_fraction=nan)
all invalid: StageIIIDiagnostics(total_particles=2, valid_particles=0, invalid_particles=2, total_weight=0.0, effective_sample_size=0.0, valid_particle_fraction=0.0)
zero weight: StageIIIDiagnostics(total_particles=2, valid_particles=2, invalid_particles=0, total_weight=0.0, effective_sample_size=0.0, valid_particle_fraction=1.0)
non-finite on invalid particle (ignored): StageIIIDiagnostics(total_particles=2, valid_particles=1, invalid_particles=1, total_weight=2.0, effective_sample_size=1.0, valid_particle_fraction=0.5)
non-finite on valid particle correctly raised: non-finite weight (NaN or Inf) found among particles marked valid -- diagnostics require every valid particle's weight to be finite; fix the upstream source of the non-finite weight rather than silently ignoring it (a non-finite weight on an INVALID particle is fine

## 5. Log-weight convenience wrapper (matches T05/T06/T09's log-space convention)

In [6]:
# log_weight = [0, ln(2), ln(3)] -> linear weights [1, 2, 3]; third particle invalid.
log_weights = np.array([0.0, math.log(2.0), math.log(3.0)])
valid_d = np.array([True, True, False])
diag_d = compute_stage3_diagnostics_from_log_weights(log_weights, valid_d)

expected_total_weight = 1.0 + 2.0  # exp(0) + exp(ln 2); exp(ln 3) excluded (invalid)
expected_ess = (1.0 + 2.0) ** 2 / (1.0 ** 2 + 2.0 ** 2)
print(f"expected total_weight={expected_total_weight}, ESS={expected_ess}")
print("actual:", diag_d)
assert math.isclose(diag_d.total_weight, expected_total_weight)
assert math.isclose(diag_d.effective_sample_size, expected_ess)
print("OK")

expected total_weight=3.0, ESS=1.8
actual: StageIIIDiagnostics(total_particles=3, valid_particles=2, invalid_particles=1, total_weight=3.0, effective_sample_size=1.8, valid_particle_fraction=0.6666666666666666)
OK


## Summary / known limitations

- `iste.diagnostics.compute_stage3_diagnostics` is the single place the Stage III
  `total/valid/invalid_particles`, `total_weight`, `effective_sample_size`, and
  `valid_particle_fraction` arithmetic is defined. `iste.ensemble_run.compute_diagnostics`/
  `compute_diagnostics_by_member` (T09) now delegate to it instead of duplicating the ESS
  formula.
- Effective sample size is Kish's standard formula `(sum w)^2 / sum(w^2)`, computed over
  **valid particles' weights only** — an invalid particle's weight never contributes,
  regardless of its value.
- A non-finite (`NaN`/`Inf`) weight on a particle marked **valid** raises `ValueError`
  rather than silently producing a misleading number; the same value on an **invalid**
  particle is harmless and ignored.
- This module does not run trajectories, does not touch T05/T06/T09's propagation logic,
  and does not implement KDE, the observation likelihood, SMC, MCMC, or AIS.
- No T11 (storage), T12 (controlled validation suite), or T13 (sign-off) work exists in
  this repository.